In [4]:
import pandas as pd
from pathlib import Path

# Ordner anlegen
Path("../data/bronze").mkdir(parents=True, exist_ok=True)
Path("../data/silver").mkdir(parents=True, exist_ok=True)
Path("../data/gold").mkdir(parents=True, exist_ok=True)


In [3]:
# Rohdaten laden
df = pd.read_excel(
    "../data/Online Retail.xlsx",
    parse_dates = ["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Description": "string",
        "CustomerID": "string",
        "Country": "string",
    }
)

# print("Raw shape:", df.shape)
# print(df.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice CustomerID         Country  
0 2010-12-01 08:26:00       2.55      17850  United Kingdom  
1 2010-12-01 08:26:00       3.39      17850  United Kingdom  
2 2010-12-01 08:26:00       2.75      17850  United Kingdom  
3 2010-12-01 08:26:00       3.39      17850  United Kingdom  
4 2010-12-01 08:26:00       3.39      17850  United Kingdom  


In [5]:
# Bronze speichern
df.to_parquet("../data/bronze/online_retail_raw.parquet", engine="pyarrow")

In [7]:
# df.isnull().sum()

In [8]:
# Quality Checks
total_rows = len(df)

returns_count = (df["Quantity"] < 0).sum()
negative_price_count = (df["UnitPrice"] <0).sum()
missing_description_count = df["Description"].isna().sum()
missing_customer_count = df["CustomerID"].isna().sum()

print("Returns:", returns_count, f"({returns_count/total_rows:.2%})")
print("Negative prices:", negative_price_count, f"({negative_price_count/total_rows:.2%})")
print("Missing descriptions:", missing_description_count, f"({missing_description_count/total_rows:.2%})")
print("Missing customer IDs:", missing_customer_count, f"({missing_customer_count/total_rows:.2%})")


Returns: 10624 (1.96%)
Negative prices: 2 (0.00%)
Missing descriptions: 1454 (0.27%)
Missing customer IDs: 135080 (24.93%)


In [9]:
# Silver Layer
sales = df[df["Quantity"] > 0].copy()
sales = sales[sales["UnitPrice"] > 0].copy()
sales = sales[sales["Description"].notna()].copy()
sales = sales[sales["CustomerID"].notna()].copy()

sales["Description"] = (
    sales["Description"]
    .str.upper()
    .str.strip()
)

# Kennzahl berechnen
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
#df[["Quantity", "UnitPrice", "Revenue"]].head()

In [10]:
# Zeitfelder erzeugen
sales["Year"] = sales["InvoiceDate"].dt.year
sales["Month"] = sales["InvoiceDate"].dt.month
sales["Day"] = sales["InvoiceDate"].dt.day
sales["Weekday"] = sales["InvoiceDate"].dt.day_name()

In [11]:
# Zwischenstand speichern (Silver Layer)
sales.to_parquet("../data/silver/online_retail_clean.parquet", engine="pyarrow")

In [12]:
# Check auf Zeilenverlust im Silver Layer
print("Raw rows:", len(df))
print("Silver rows:", len(sales))
print("Rows removed:", len(df) - len(sales))

Raw rows: 541909
Silver rows: 397884
Rows removed: 144025


In [74]:
# print("Silver shape:", sales.shape)

In [13]:
# Product Consistency Checks
product_desc_check = (
    sales.groupby("StockCode")["Description"].nunique()
)

product_desc_issues = product_desc_check[product_desc_check > 1]

if len(product_desc_issues) > 0:
    print("Products with multiple descriptions:", len(product_desc_issues))
    print(product_desc_issues.head())

# Customer consistency check
customer_country_check = (
    sales.groupby("CustomerID")["Country"].nunique()
)

customer_country_issues = customer_country_check[customer_country_check > 1]

if len(customer_country_issues) > 0:
    print("Customers with multiple countries:", len(customer_country_issues))
    print(customer_country_issues.head())

Products with multiple descriptions: 210
StockCode
16156L    2
17107D    3
20622     2
20725     2
20914     2
Name: Description, dtype: int64
Customers with multiple countries: 8
CustomerID
12370    2
12394    2
12417    2
12422    2
12429    2
Name: Country, dtype: int64


In [18]:
# dim_product erstellen
# Zählen wie oft jede Description vorkommt, nach Häufigkeit sortieren und die häufigste Description pro StockCode nehmen
dim_product = (
    sales.groupby(["StockCode", "Description"])
    .size()
    .reset_index(name = "count")
    .sort_values(["StockCode", "count"], ascending = [True, False])
    .drop_duplicates(subset=["StockCode"])
    .drop(columns = "count")
    .reset_index(drop = True)
)
# ProductKey erzeugen
dim_product["ProductKey"] = dim_product.index + 1 
dim_product = dim_product[
    ["ProductKey", "StockCode", "Description"]
]

In [19]:
# dim_customer erstellen
dim_customer = (
    sales.groupby(["CustomerID", "Country"])
    .size()
    .reset_index(name = "count")
    .sort_values(["CustomerID", "count"], ascending = [True, False])
    .drop_duplicates(subset=["CustomerID"])
    .drop(columns = "count")
    .reset_index(drop=True)
)
# CustomerKey erzeugen
dim_customer["CustomerKey"] = dim_customer.index + 1

dim_customer = dim_customer[
    ["CustomerKey", "CustomerID", "Country"]
]

# dim_customer.head()

,CustomerKey,CustomerID,Country
0,1,12346,United Kingdom
1,2,12347,Iceland
2,3,12348,Finland
3,4,12349,Italy
4,5,12350,Norway


In [20]:
# dim_date erstellen
date_range = pd.date_range(
    start=sales["InvoiceDate"].min().normalize(),
    end=sales["InvoiceDate"].max().normalize(),
    freq="D"
)

dim_date = pd.DataFrame({"Date": date_range})

# DateKey erzeugen
dim_date["DateKey"] = dim_date["Date"].dt.strftime("%Y%m%d").astype(int)
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Quarter"] = dim_date["Date"].dt.quarter

dim_date["MonthNumber"] = dim_date["Date"].dt.month
dim_date["MonthName"] = dim_date["Date"].dt.month_name()

dim_date["Day"] = dim_date["Date"].dt.day

dim_date["YearMonth"] = dim_date["Date"].dt.strftime("%Y-%m")
dim_date["Weekday"] = dim_date["Date"].dt.day_name()
dim_date["WeekdayNumber"] = dim_date["Date"].dt.weekday
dim_date["IsWeekend"] = dim_date["WeekdayNumber"].isin([5,6])

dim_date = dim_date[
    ["DateKey", "Date", "Year", "Quarter", "MonthNumber", "MonthName", "YearMonth", "Day", "Weekday", "WeekdayNumber", "IsWeekend"]
]

# dim_date.head()

,DateKey,Date,Year,Quarter,MonthNumber,MonthName,YearMonth,Day,Weekday,WeekdayNumber,IsWeekend
0,20101201,2010-12-01,2010,4,12,December,2010-12,1,Wednesday,2,False
1,20101202,2010-12-02,2010,4,12,December,2010-12,2,Thursday,3,False
2,20101203,2010-12-03,2010,4,12,December,2010-12,3,Friday,4,False
3,20101204,2010-12-04,2010,4,12,December,2010-12,4,Saturday,5,True
4,20101205,2010-12-05,2010,4,12,December,2010-12,5,Sunday,6,True


In [21]:
# fact_sales erstellen
fact_sales = sales[
    [
    "InvoiceNo",
    "StockCode",
    "CustomerID",
    "InvoiceDate",
    "Quantity",
    "UnitPrice",
    "Revenue"
    ]
].copy()


In [22]:
# fact_sales mit Keys verbinden
fact_sales = fact_sales.merge(
    dim_product[["ProductKey", "StockCode"]],
    on = "StockCode",
    how ="left"
)


In [23]:
fact_sales = fact_sales.merge(
    dim_customer[["CustomerKey", "CustomerID"]],
    on = "CustomerID",
    how = "left"
)

In [24]:
fact_sales["DateKey"] = fact_sales["InvoiceDate"].dt.floor("D").dt.strftime("%Y%m%d").astype(int)

In [25]:
fact_sales = fact_sales[
    [
        "InvoiceNo",
        "DateKey",
        "ProductKey",
        "CustomerKey",
        "Quantity",
        "UnitPrice",
        "Revenue",
    ]
].copy()
# fact_sales.head()

,InvoiceNo,DateKey,ProductKey,CustomerKey,Quantity,UnitPrice,Revenue
0,536365,20101201,3234,4017,6,2.55,15.30
1,536365,20101201,2644,4017,6,3.39,20.34
2,536365,20101201,2848,4017,8,2.75,22.00
3,536365,20101201,2796,4017,6,3.39,20.34
4,536365,20101201,2795,4017,6,3.39,20.34


In [26]:
# fact_sales[["ProductKey", "CustomerKey", "DateKey"]].isna().sum()

ProductKey     0
CustomerKey    0
DateKey        0
dtype: int64

In [27]:
# Data Quality Checks 
print("Sales shape:", sales.shape)
print("fact_sales shape:", fact_sales.shape)

# Pipeline Integrity: Row Count Check Gehen Zeilen verloren bei der Transformation?
assert len(sales) == len(fact_sales), "Row count mismatch between sales and fact_sales"

# Foreign Key Check
assert fact_sales["ProductKey"].isna().sum() == 0, "Missing ProductKey detected"
assert fact_sales["CustomerKey"].isna().sum() == 0, "Missing CustomerKey detected"
assert fact_sales["DateKey"].isna().sum() == 0, "Missing DateKey detected"

# Business Rules
assert fact_sales["Quantity"].min() > 0 # keine Returns
assert fact_sales["UnitPrice"].min() > 0 # keine negativen Preise

# Dimension Checks
assert dim_product["StockCode"].duplicated().sum() == 0
assert dim_customer["CustomerID"].duplicated().sum() == 0
assert dim_date["DateKey"].duplicated().sum() == 0

Sales shape: (397884, 13)
fact_sales shape: (397884, 7)


In [28]:
dim_product = dim_product.sort_values("ProductKey").reset_index(drop=True)
dim_customer = dim_customer.sort_values("CustomerKey").reset_index(drop=True)
dim_date = dim_date.sort_values("DateKey").reset_index(drop=True)
fact_sales = fact_sales.sort_values(["DateKey", "InvoiceNo"]).reset_index(drop=True)

In [29]:
dim_product.to_parquet("../data/gold/dim_product.parquet", engine = "pyarrow")
dim_customer.to_parquet("../data/gold/dim_customer.parquet", engine = "pyarrow")
dim_date.to_parquet("../data/gold/dim_date.parquet", engine = "pyarrow")
fact_sales.to_parquet("../data/gold/fact_sales.parquet", engine="pyarrow")